In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ============================================================
# SETTINGS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime"
)

DATASETS_DIR = PROJECT_DIR / "datasets"
CHECKPOINTS_DIR = PROJECT_DIR / "checkpoints"

SURFACE_CONFIGS = {
    1462: {
        "pickle": CHECKPOINTS_DIR / "pseudotime_cells_clean_1462.pkl",
        "outliers": DATASETS_DIR / "1462_dapi_python_segmentation_outliers.txt",
        "cellprofiler_dir": DATASETS_DIR / "1462",
        "output": DATASETS_DIR / "combined_clustered_data_1462_k2.csv",
    },
    1476: {
        "pickle": CHECKPOINTS_DIR / "pseudotime_cells_clean_1476.pkl",
        "outliers": DATASETS_DIR / "1476_dapi_python_segmentation_outliers.txt",
        "cellprofiler_dir": DATASETS_DIR / "1476",
        "output": DATASETS_DIR / "combined_clustered_data_1476_k2.csv",
    },
}

MIN_PSEUDOTIME_WIDTH_LENGTH = 6
TARGET_K = 2
N_PCS_CLUSTER = 5

DAPI_CP_FILE = "MyExpt_IdentifyPrimaryObjects.csv"
PHALLOIDIN_CP_FILE = "MyExpt_Phalloidin_segmented.csv"
YAP_CP_FILE = "MyExpt_YAP_segmented.csv"


# ============================================================
# HELPERS
# ============================================================

def safe_len(x):
    try:
        return len(x)
    except TypeError:
        return 0


def load_outlier_names(outlier_file: Path):
    if not outlier_file.exists():
        print(f"  Outlier file not found; no DAPI outliers removed: {outlier_file}")
        return set()

    outliers = []
    with open(outlier_file, "r") as f:
        for line in f:
            name = line.strip()
            if not name:
                continue
            if not name.lower().endswith(".tif"):
                name = name + ".tif"
            outliers.append(name)

    return set(outliers)


def resample_ts_raw(ts, target_len):
    ts = np.asarray(ts, dtype=float)

    if len(ts) == 0:
        raise ValueError("Cannot resample an empty pseudotime-width vector.")

    if len(ts) == 1:
        return np.full(target_len, ts[0], dtype=float)

    x_old = np.linspace(0, 1, len(ts))
    x_new = np.linspace(0, 1, target_len)
    return np.interp(x_new, x_old, ts)


def add_agglomerative_k2_cluster(
    df,
    width_col="pseudotime_widths",
    cluster_col="cluster_agg_k2",
    n_pcs_cluster=5,
):
    X_series = df[width_col].to_list()
    lengths = np.array([safe_len(ts) for ts in X_series], dtype=int)

    if len(df) < 2:
        raise ValueError("Need at least 2 rows for k=2 clustering.")

    if np.any(lengths == 0):
        raise ValueError("Found empty pseudotime-width vectors after filtering.")

    target_len = int(np.median(lengths))

    X_shape = np.vstack([
        resample_ts_raw(ts, target_len)
        for ts in X_series
    ])

    orig_len = lengths.astype(float).reshape(-1, 1)
    X_features = np.hstack([X_shape, orig_len])

    X_scaled = StandardScaler().fit_transform(X_features)

    n_pcs_use = min(
        int(n_pcs_cluster),
        X_scaled.shape[0] - 1,
        X_scaled.shape[1],
    )

    if n_pcs_use < 1:
        raise ValueError("Not enough samples/features to run PCA.")

    pca = PCA(n_components=n_pcs_use, svd_solver="full")
    X_pca = pca.fit_transform(X_scaled)

    agg = AgglomerativeClustering(n_clusters=2, linkage="ward")
    labels = agg.fit_predict(X_pca)

    out = df.copy()
    out[cluster_col] = labels

    unique, counts = np.unique(labels, return_counts=True)
    sizes = dict(zip(unique.astype(int), counts.astype(int)))

    if len(np.unique(labels)) >= 2 and min(counts) >= 2:
        sil = silhouette_score(X_pca, labels)
    else:
        sil = np.nan

    info = {
        "target_len": target_len,
        "n_pcs_used": n_pcs_use,
        "explained_variance_sum": float(pca.explained_variance_ratio_.sum()),
        "cluster_sizes": sizes,
        "silhouette": sil,
    }

    return out, info


def prepare_df(df, prefix, keys):
    missing_keys = [key for key in keys if key not in df.columns]
    if missing_keys:
        raise ValueError(f"Missing merge keys {missing_keys} in {prefix} dataframe.")

    rename_dict = {
        col: f"{prefix}_{col}"
        for col in df.columns
        if col not in keys
    }
    return df.rename(columns=rename_dict)


def load_and_merge_cellprofiler_data(SD_clean, surface_id, cellprofiler_dir: Path):
    dapi_file = cellprofiler_dir / DAPI_CP_FILE
    phalloidin_file = cellprofiler_dir / PHALLOIDIN_CP_FILE
    yap_file = cellprofiler_dir / YAP_CP_FILE

    for f in [dapi_file, phalloidin_file, yap_file]:
        if not f.exists():
            raise FileNotFoundError(f"Required CellProfiler file not found:\n{f}")

    df_identifyPrimaryObjects = pd.read_csv(dapi_file)
    df_Phalloidin_segmented = pd.read_csv(phalloidin_file)
    df_YAP_segmented = pd.read_csv(yap_file)

    SD_clean = SD_clean.copy()

    SD_clean["ImageNumber"] = SD_clean.index // 10000
    SD_clean["ObjectNumber"] = SD_clean.index % 10000

    keys = ["Metadata_ObjNumber", "ObjectNumber"]
    cluster_map = SD_clean.rename(columns={"ImageNumber": "Metadata_ObjNumber"})

    df_id = prepare_df(df_identifyPrimaryObjects, "dapi", keys)
    df_ph = prepare_df(df_Phalloidin_segmented, "phalloidin", keys)
    df_yap = prepare_df(df_YAP_segmented, "yap", keys)

    merged = (
        cluster_map
        .merge(df_id, on=keys, how="inner")
        .merge(df_ph, on=keys, how="inner")
        .merge(df_yap, on=keys, how="inner")
    )

    merged["Feature_Idx"] = int(surface_id)

    before = len(merged)
    merged = merged[
        merged["yap_AreaShape_Area"] > merged["dapi_AreaShape_Area"]
    ].copy()
    removed = before - len(merged)

    return merged, removed


def build_combined_clustered_surface(surface_id, cfg):
    print("\n" + "=" * 80)
    print(f"Processing surface {surface_id}")

    pickle_file = Path(cfg["pickle"])
    outlier_file = Path(cfg["outliers"])
    cellprofiler_dir = Path(cfg["cellprofiler_dir"])
    output_file = Path(cfg["output"])

    if not pickle_file.exists():
        raise FileNotFoundError(f"Pseudotime pickle not found:\n{pickle_file}")

    SD_clean = pd.read_pickle(pickle_file)
    print(f"  Loaded pseudotime dataframe: {SD_clean.shape[0]} rows × {SD_clean.shape[1]} columns")

    if "pseudotime_widths" not in SD_clean.columns:
        raise ValueError("Expected column 'pseudotime_widths' was not found.")

    if "path_dapi" not in SD_clean.columns:
        raise ValueError("Expected column 'path_dapi' was not found.")

    before_len_filter = len(SD_clean)
    SD_clean = SD_clean[
        SD_clean["pseudotime_widths"].apply(safe_len) >= MIN_PSEUDOTIME_WIDTH_LENGTH
    ].copy()
    print(
        f"  Removed {before_len_filter - len(SD_clean)} rows with "
        f"pseudotime_widths shorter than {MIN_PSEUDOTIME_WIDTH_LENGTH}"
    )

    outliers = load_outlier_names(outlier_file)
    if outliers:
        before_outliers = len(SD_clean)
        SD_clean = SD_clean[
            ~SD_clean["path_dapi"].apply(lambda p: Path(p).name).isin(outliers)
        ].copy()
        print(f"  Removed {before_outliers - len(SD_clean)} DAPI outlier rows")

    SD_clean, clustering_info = add_agglomerative_k2_cluster(
        SD_clean,
        width_col="pseudotime_widths",
        cluster_col=f"cluster_agg_k{TARGET_K}",
        n_pcs_cluster=N_PCS_CLUSTER,
    )

    print(f"  PCA target_len: {clustering_info['target_len']}")
    print(f"  PCA components used: {clustering_info['n_pcs_used']}")
    print(f"  PCA explained variance sum: {clustering_info['explained_variance_sum']:.3f}")
    print(f"  k=2 cluster sizes: {clustering_info['cluster_sizes']}")
    print(f"  k=2 silhouette: {clustering_info['silhouette']:.3f}")

    SD_combined, removed_no_cyto = load_and_merge_cellprofiler_data(
        SD_clean=SD_clean,
        surface_id=surface_id,
        cellprofiler_dir=cellprofiler_dir,
    )

    print(f"  Removed {removed_no_cyto} objects where YAP area <= DAPI area")
    print(f"  Final combined dataframe: {SD_combined.shape[0]} rows × {SD_combined.shape[1]} columns")

    output_file.parent.mkdir(parents=True, exist_ok=True)
    SD_combined.to_csv(output_file, index=False)
    print(f"  Saved combined clustered data:\n  {output_file}")

    return output_file, SD_combined


# ============================================================
# RUN FOR BOTH SURFACES
# ============================================================

combined_clustered_files = []

for surface_id, cfg in SURFACE_CONFIGS.items():
    output_file, _ = build_combined_clustered_surface(surface_id, cfg)
    combined_clustered_files.append(output_file)

print("\nFinished generating combined clustered k=2 files:")
for f in combined_clustered_files:
    print(f"  - {f}")


Processing surface 1462
  Loaded pseudotime dataframe: 689 rows × 293 columns
  Removed 6 rows with pseudotime_widths shorter than 6
  Removed 27 DAPI outlier rows
  PCA target_len: 16
  PCA components used: 5
  PCA explained variance sum: 0.839
  k=2 cluster sizes: {np.int64(0): np.int64(344), np.int64(1): np.int64(312)}
  k=2 silhouette: 0.271
  Removed 4 objects where YAP area <= DAPI area
  Final combined dataframe: 410 rows × 1961 columns
  Saved combined clustered data:
  C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\combined_clustered_data_1462_k2.csv

Processing surface 1476
  Loaded pseudotime dataframe: 927 rows × 293 columns
  Removed 18 rows with pseudotime_widths shorter than 6
  Removed 31 DAPI outlier rows
  PCA target_len: 15
  PCA components used: 5
  PCA explained variance sum: 0.898
  k=2 cluster sizes: {np.int64(0): np.int64(569), np.int64(1): np.int64(309)}
  k=2 silhouette: 0.343
  Removed 0 objects where YAP area <= DAPI area
  

In [2]:
from pathlib import Path
import re
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets")
REFERENCE_FILE = BASE_DIR / r"for Danya ML\merged_filtered_data_for_Danya_clean.csv"

INPUT_FILES = [
    BASE_DIR / "combined_clustered_data_1462_k2.csv",
    BASE_DIR / "combined_clustered_data_1476_k2.csv",
]

OUTPUT_DIR = BASE_DIR / "aligned_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Allowed missing columns that we can reconstruct automatically
AUTO_CREATE_COLUMNS = {"Feature_Idx"}

# ============================================================
# HELPERS
# ============================================================

def read_columns(csv_path: Path):
    return list(pd.read_csv(csv_path, nrows=0).columns)


def find_canonical_cluster_column(reference_columns):
    """
    Find the exact canonical cluster column name in the reference.
    This preserves trailing spaces if they exist in the reference header.
    """
    matches = [col for col in reference_columns if col.strip() == "cluster_agg_k"]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one cluster_agg_k-like column in reference, found {len(matches)}: {matches}"
        )
    return matches[0]


def find_cluster_variant(columns):
    """
    Accept:
      cluster_agg_k
      cluster_agg_k2
      cluster_agg_k3
      cluster_agg_k5
      cluster_agg_k12
    and also tolerate trailing spaces.
    """
    pattern = re.compile(r"^cluster_agg_k\d*$")
    matches = [col for col in columns if pattern.fullmatch(col.strip())]

    if len(matches) == 0:
        return None
    if len(matches) > 1:
        raise ValueError(f"More than one cluster-like column found: {matches}")
    return matches[0]


def infer_feature_idx_from_filename(file_path: Path):
    """
    Extract Feature_Idx from names like:
      combined_clustered_data_1462_k5.csv
      combined_clustered_data_1476_k3.csv
    """
    m = re.search(r"combined_clustered_data_(\d+)_k\d+", file_path.stem)
    if m:
        return int(m.group(1))

    # fallback: first standalone 3+ digit number in filename
    m = re.search(r"(?<!\d)(\d{3,})(?!\d)", file_path.stem)
    if m:
        return int(m.group(1))

    raise ValueError(
        f"Could not infer Feature_Idx from filename: {file_path.name}"
    )


def build_missing_columns(df, missing_columns, input_file):
    """
    Create allowed missing columns.
    Currently supports Feature_Idx from filename.
    """
    for col in missing_columns:
        if col == "Feature_Idx":
            feature_idx = infer_feature_idx_from_filename(input_file)
            df[col] = feature_idx
            print(f"  Added missing column {repr(col)} with value {feature_idx} from filename.")
        else:
            raise ValueError(f"No auto-create rule implemented for missing column: {col}")

    return df


def align_file_to_reference(reference_file: Path, input_file: Path, output_dir: Path):
    print(f"\nProcessing: {input_file.name}")

    reference_columns = read_columns(reference_file)
    canonical_cluster_name = find_canonical_cluster_column(reference_columns)

    df = pd.read_csv(input_file)
    original_columns = list(df.columns)

    # --------------------------------------------------------
    # 1. Rename cluster column variant to exact reference name
    # --------------------------------------------------------
    cluster_variant = find_cluster_variant(original_columns)
    if cluster_variant is not None and cluster_variant != canonical_cluster_name:
        df = df.rename(columns={cluster_variant: canonical_cluster_name})
        print(f"  Renamed special column: {repr(cluster_variant)} -> {repr(canonical_cluster_name)}")

    current_columns = list(df.columns)

    # --------------------------------------------------------
    # 2. Detect missing and extra columns
    # --------------------------------------------------------
    missing = [col for col in reference_columns if col not in current_columns]
    extra = [col for col in current_columns if col not in reference_columns]

    if missing:
        print(f"  Missing columns before repair ({len(missing)}): {missing}")

    if extra:
        print(f"  Extra columns to drop ({len(extra)}).")
        for col in extra[:20]:
            print(f"    - {repr(col)}")
        if len(extra) > 20:
            print(f"    ... and {len(extra) - 20} more")

    # --------------------------------------------------------
    # 3. Reconstruct allowed missing columns
    # --------------------------------------------------------
    disallowed_missing = [col for col in missing if col not in AUTO_CREATE_COLUMNS]
    allowed_missing = [col for col in missing if col in AUTO_CREATE_COLUMNS]

    if disallowed_missing:
        raise ValueError(
            f"{input_file.name} is still missing required columns that cannot be auto-created: {disallowed_missing}"
        )

    if allowed_missing:
        df = build_missing_columns(df, allowed_missing, input_file)

    # --------------------------------------------------------
    # 4. Drop extras so final schema matches reference exactly
    # --------------------------------------------------------
    if extra:
        df = df.drop(columns=extra)

    # --------------------------------------------------------
    # 5. Reorder exactly to reference
    # --------------------------------------------------------
    final_missing = [col for col in reference_columns if col not in df.columns]
    final_extra = [col for col in df.columns if col not in reference_columns]

    if final_missing or final_extra:
        raise ValueError(
            f"After harmonization, schema still does not match for {input_file.name}. "
            f"Missing: {final_missing}; Extra: {final_extra}"
        )

    df = df.loc[:, reference_columns]

    # --------------------------------------------------------
    # 6. Final strict verification
    # --------------------------------------------------------
    final_columns = list(df.columns)
    if final_columns != reference_columns:
        for i, (ref_col, final_col) in enumerate(zip(reference_columns, final_columns)):
            if ref_col != final_col:
                raise ValueError(
                    f"Column order mismatch at position {i} in {input_file.name}: "
                    f"reference={repr(ref_col)}, final={repr(final_col)}"
                )

    output_file = output_dir / f"{input_file.stem}_aligned.csv"
    df.to_csv(output_file, index=False)

    print(f"  Saved: {output_file}")
    print(f"  Final column count: {len(final_columns)}")
    return output_file


def verify_all_outputs_identical(reference_file: Path, aligned_files):
    """
    Verify that all final output files have exactly the same header
    as the reference, in the same order.
    """
    reference_columns = read_columns(reference_file)

    print("\n================ FINAL VERIFICATION ================")
    for f in aligned_files:
        cols = read_columns(f)

        if cols != reference_columns:
            raise ValueError(f"{f.name} does NOT match the reference header exactly.")
        print(f"[OK] {f.name} matches the reference exactly.")

    print("\nAll final databases have identical columns and identical order.")


# ============================================================
# RUN
# ============================================================

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f"Reference file not found:\n{REFERENCE_FILE}")

for f in INPUT_FILES:
    if not f.exists():
        raise FileNotFoundError(f"Input file not found:\n{f}")

aligned_files = []
for input_file in INPUT_FILES:
    aligned_file = align_file_to_reference(
        reference_file=REFERENCE_FILE,
        input_file=input_file,
        output_dir=OUTPUT_DIR
    )
    aligned_files.append(aligned_file)

verify_all_outputs_identical(REFERENCE_FILE, aligned_files)


Processing: combined_clustered_data_1462_k2.csv
  Renamed special column: 'cluster_agg_k2' -> 'cluster_agg_k'
  Extra columns to drop (434).
    - 'path_dapi'
    - 'path_yap'
    - 'path_actin'
    - 'height'
    - 'width'
    - 'channels'
    - 'radial_distribution.RadialDistribution_FracAtD_1of4'
    - 'radial_distribution.RadialDistribution_MeanFrac_1of4'
    - 'radial_distribution.RadialDistribution_RadialCV_1of4'
    - 'radial_distribution.RadialDistribution_FracAtD_2of4'
    - 'radial_distribution.RadialDistribution_MeanFrac_2of4'
    - 'radial_distribution.RadialDistribution_RadialCV_2of4'
    - 'radial_distribution.RadialDistribution_FracAtD_3of4'
    - 'radial_distribution.RadialDistribution_MeanFrac_3of4'
    - 'radial_distribution.RadialDistribution_RadialCV_3of4'
    - 'radial_distribution.RadialDistribution_FracAtD_4of4'
    - 'radial_distribution.RadialDistribution_MeanFrac_4of4'
    - 'radial_distribution.RadialDistribution_RadialCV_4of4'
    - 'radial_zernikes.RadialD

In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets")
ALIGNED_DIR = BASE_DIR / "aligned_output"

ALIGNED_FILES = [
    ALIGNED_DIR / "combined_clustered_data_1462_k2_aligned.csv",
    ALIGNED_DIR / "combined_clustered_data_1476_k2_aligned.csv",
]

MERGED_OUTPUT_FILE = ALIGNED_DIR / "merged_aligned_databases_k2_both_surfaces.csv"

# Optional: keep track of which file each row came from
ADD_SOURCE_FILE_COLUMN = True


# ============================================================
# HELPERS
# ============================================================

def read_columns(csv_path):
    return list(pd.read_csv(csv_path, nrows=0).columns)


# ============================================================
# CHECK FILES EXIST
# ============================================================

for f in ALIGNED_FILES:
    if not f.exists():
        raise FileNotFoundError(f"Aligned file not found:\n{f}")

print("All aligned files found.")


# ============================================================
# VERIFY COLUMN IDENTITY
# ============================================================

reference_columns = read_columns(ALIGNED_FILES[0])

for f in ALIGNED_FILES[1:]:
    cols = read_columns(f)
    if cols != reference_columns:
        missing = [c for c in reference_columns if c not in cols]
        extra = [c for c in cols if c not in reference_columns]

        raise ValueError(
            f"Column mismatch in {f.name}\n"
            f"Missing columns: {missing}\n"
            f"Extra columns: {extra}"
        )

print(f"All files have identical columns in identical order.")
print(f"Column count: {len(reference_columns)}")


# ============================================================
# LOAD AND MERGE
# ============================================================

dfs = []

for f in ALIGNED_FILES:
    df = pd.read_csv(f)

    if ADD_SOURCE_FILE_COLUMN:
        df["source_file"] = f.name

    dfs.append(df)
    print(f"Loaded {f.name}: {df.shape[0]} rows × {df.shape[1]} columns")

merged_df = pd.concat(dfs, axis=0, ignore_index=True)

print("\nMerged dataframe shape:", merged_df.shape)


# ============================================================
# SAVE MERGED CSV
# ============================================================

merged_df.to_csv(MERGED_OUTPUT_FILE, index=False)

print(f"\nSaved merged CSV to:\n{MERGED_OUTPUT_FILE}")

All aligned files found.
All files have identical columns in identical order.
Column count: 1527
Loaded combined_clustered_data_1462_k2_aligned.csv: 410 rows × 1528 columns
Loaded combined_clustered_data_1476_k2_aligned.csv: 667 rows × 1528 columns

Merged dataframe shape: (1077, 1528)

Saved merged CSV to:
C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\aligned_output\merged_aligned_databases_k2_both_surfaces.csv
